# 🏆 Time Series Competition — v3 (Walk-Forward CV)

## Tại sao v3 khác v1 (0.1919)?

| | v1 (0.1919) | v3 |
|---|---|---|
| Validation split | 1 fold 90/10 | Walk-Forward 3 folds |
| Clip percentile | P0.5 / P99.5 | P0.1 / P99.9 + buffer |
| max_bin lỗi | Chưa gặp | Fix: max_bin → Dataset.params |
| Public/Private correlation | 1 ước tính | 3 ước tính (fold3 ≈ Public, mean ≈ Private) |

## Walk-Forward CV
```
[==TRAIN 70%==|=VAL 10%=|          ]  fold 1
[====TRAIN 80%====|=VAL 10%=|      ]  fold 2  
[======TRAIN 90%======|=VAL 10%=   ]  fold 3  ← gần test nhất
```

## Fix lỗi max_bin
```python
# ❌ SAI — gây lỗi khi dùng với .construct() hoặc có Dataset trước
params = {'max_bin': 127, ...}

# ✅ ĐÚNG — max_bin đặt trong Dataset, KHÔNG trong lgb.train params
lgb.Dataset(..., params={'max_bin': 127})
```

## Cell 1 — Cài đặt

In [ ]:
import subprocess, sys
for p in ['polars', 'lightgbm', 'psutil']:
    try:
        __import__(p)
    except ImportError:
        print(f'Installing {p}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', p, '-q'])
print('All packages ready')

## Cell 2 — Imports & Config

In [ ]:
import numpy as np
import polars as pl
import lightgbm as lgb
import pandas as pd
import gc, os, warnings, psutil
warnings.filterwarnings('ignore')

def ram_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1e6

print(f"LightGBM : {lgb.__version__}")
print(f"Polars   : {pl.__version__}")
print(f"NumPy    : {np.__version__}")
print(f"RAM      : {ram_mb():.0f} MB / {psutil.virtual_memory().total/1e9:.1f} GB")

# ─── PATHS ───
TRAIN_PATH  = "/kaggle/input/ts-forecasting/train.parquet"
TEST_PATH   = "/kaggle/input/ts-forecasting/test.parquet"
OUTPUT_PATH = "/kaggle/working/submission_v3.csv"

RANDOM_SEED           = 42
NUM_BOOST_ROUND       = 3000
EARLY_STOPPING_ROUNDS = 150
EXPECTED_TEST_ROWS    = 1_447_107
PRED_BATCH_SIZE       = 200_000

# Walk-Forward: (train_end_pct, val_end_pct)
WF_FOLDS = [
    (0.70, 0.80),
    (0.80, 0.90),
    (0.90, 1.00),
]

# Clip rộng hơn v1 — P0.1/P99.9 + 10% buffer
CLIP_LOW_PCTILE  = 0.1
CLIP_HIGH_PCTILE = 99.9
CLIP_BUFFER      = 0.10

## Cell 3 — Metric

In [ ]:
def weighted_rmse_score(y_target, y_pred, w):
    denom = np.sum(w * y_target ** 2)
    if denom == 0:
        return 0.0
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    return float(np.sqrt(1.0 - np.clip(ratio, 0.0, 1.0)))

print('Metric ready')

## Cell 4 — Xác định cột

In [ ]:
schema   = pl.scan_parquet(TRAIN_PATH).schema
all_cols = list(schema.keys())

META_COLS    = ['id', 'code', 'sub_code', 'sub_category', 'horizon', 'ts_index']
TARGET_COL   = 'y_target'
WEIGHT_COL   = 'weight'
CAT_COLS     = ['code', 'sub_code', 'sub_category']
FEATURE_COLS = [c for c in all_cols if c.startswith('feature_')]
NEEDED_COLS  = [c for c in META_COLS + FEATURE_COLS + [TARGET_COL, WEIGHT_COL] if c in all_cols]

print(f"Feature cols : {len(FEATURE_COLS)}")
print(f"Cols to load : {len(NEEDED_COLS)}")

## Cell 5 — Load Train (Lazy + F32)

In [ ]:
print(f"Loading train.parquet... RAM: {ram_mb():.0f} MB")

train = (
    pl.scan_parquet(TRAIN_PATH)
    .select(NEEDED_COLS)
    .with_columns(
        [pl.col(c).cast(pl.Float32) for c in FEATURE_COLS] +
        [pl.col(TARGET_COL).cast(pl.Float32),
         pl.col(WEIGHT_COL).cast(pl.Float32),
         pl.col('horizon').cast(pl.Int16),
         pl.col('ts_index').cast(pl.Int32)]
    )
    .collect()
)
gc.collect()
print(f"Train: {train.shape}  {train.estimated_size('mb'):.0f}MB  RAM:{ram_mb():.0f}MB")

# Tính clip range ngay từ y_target
y_all       = train[TARGET_COL].to_numpy()
clip_lo_raw = float(np.percentile(y_all, CLIP_LOW_PCTILE))
clip_hi_raw = float(np.percentile(y_all, CLIP_HIGH_PCTILE))
clip_rng    = clip_hi_raw - clip_lo_raw
CLIP_LOW    = clip_lo_raw - CLIP_BUFFER * clip_rng
CLIP_HIGH   = clip_hi_raw + CLIP_BUFFER * clip_rng
del y_all; gc.collect()

print(f"y_target clip: [{CLIP_LOW:.4f}, {CLIP_HIGH:.4f}]  (P{CLIP_LOW_PCTILE}/P{CLIP_HIGH_PCTILE} + {CLIP_BUFFER*100:.0f}% buffer)")

## Cell 6 — Imputation

In [ ]:
impute_vals = {}
for c in FEATURE_COLS:
    s = train[c].drop_nulls()
    try:
        s = s.drop_nans()
    except Exception:
        pass
    med = s.median()
    impute_vals[c] = float(med) if med is not None else 0.0

train = train.with_columns([
    pl.col(c).fill_null(impute_vals[c]).fill_nan(impute_vals[c])
    for c in FEATURE_COLS
])
print(f"NaN remaining: {sum(train[c].is_null().sum()+train[c].is_nan().sum() for c in FEATURE_COLS)}")
print(f"RAM: {ram_mb():.0f} MB")

## Cell 7 — Label Encoding

In [ ]:
cat_maps = {}
ENC_COLS = []

for col in CAT_COLS:
    uniques  = sorted(train[col].cast(pl.Utf8).unique().to_list())
    cat_map  = {v: i for i, v in enumerate(uniques)}
    cat_maps[col] = cat_map
    enc = col + '_enc'
    ENC_COLS.append(enc)
    train = train.with_columns(
        pl.col(col).cast(pl.Utf8).replace(cat_map, default=-1).cast(pl.Int16).alias(enc)
    )
    print(f"  {col}: {len(uniques)} unique")

print(f"ENC_COLS: {ENC_COLS}  RAM: {ram_mb():.0f} MB")

## Cell 8 — Polars → NumPy

In [ ]:
ALL_FEAT = FEATURE_COLS + ENC_COLS + ['horizon', 'ts_index']
print(f"Features: {len(ALL_FEAT)}  |  RAM before: {ram_mb():.0f} MB")

X_arr  = train.select(ALL_FEAT).to_numpy(allow_copy=True).astype(np.float32)
y_arr  = train[TARGET_COL].to_numpy().astype(np.float32)
w_arr  = train[WEIGHT_COL].to_numpy().astype(np.float32)
ts_arr = train['ts_index'].to_numpy().astype(np.int32)

del train; gc.collect()
print(f"X_arr: {X_arr.shape}  {X_arr.nbytes/1e6:.0f}MB  |  RAM: {ram_mb():.0f} MB")

## Cell 9 — LightGBM Parameters

**max_bin fix**: Đặt trong `DATASET_PARAMS` truyền vào `lgb.Dataset()`, KHÔNG trong `LGB_PARAMS`

In [ ]:
import subprocess
try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
    GPU_OK = r.returncode == 0
except Exception:
    GPU_OK = False
DEVICE = 'cuda' if GPU_OK else 'cpu'
print(f"Device: {DEVICE}")

LGB_PARAMS = {
    'objective'               : 'regression',
    'metric'                  : 'rmse',
    'boosting_type'           : 'gbdt',
    'num_leaves'              : 63,
    'max_depth'               : 6,
    'min_data_in_leaf'        : 500,
    'min_sum_hessian_in_leaf' : 10.0,
    'min_gain_to_split'       : 0.01,
    'lambda_l1'               : 1.0,
    'lambda_l2'               : 1.0,
    'feature_fraction'        : 0.70,
    'bagging_fraction'        : 0.80,
    'bagging_freq'            : 5,
    'learning_rate'           : 0.05,
    'max_cat_threshold'       : 256,
    'device_type'             : DEVICE,
    'gpu_platform_id'         : 0,
    'gpu_device_id'           : 0,
    'verbose'                 : -1,
    'seed'                    : RANDOM_SEED,
    'num_threads'             : -1,
    # max_bin KHÔNG đặt ở đây
}

# max_bin đặt riêng cho Dataset — fix lỗi 'Cannot change max_bin after constructed Dataset'
DATASET_PARAMS = {'max_bin': 127, 'histogram_pool_size': 512}

cat_indices = [ALL_FEAT.index(c) for c in ENC_COLS]
print(f"LGB_PARAMS ready  |  DATASET_PARAMS: {DATASET_PARAMS}")

## Cell 10 — Walk-Forward Cross-Validation

In [ ]:
ts_min  = int(ts_arr.min())
ts_max  = int(ts_arr.max())
ts_span = ts_max - ts_min

print(f"ts_index: [{ts_min}, {ts_max}]  span={ts_span}")
print(f"Walk-Forward CV — {len(WF_FOLDS)} folds\n")

fold_best_iters = []
fold_val_scores = []

for fold_idx, (train_end_pct, val_end_pct) in enumerate(WF_FOLDS):
    fold_num     = fold_idx + 1
    ts_tr_end    = ts_min + ts_span * train_end_pct
    ts_vl_end    = ts_min + ts_span * val_end_pct

    train_mask   = ts_arr <= ts_tr_end
    val_mask     = (ts_arr > ts_tr_end) & (ts_arr <= ts_vl_end)

    n_tr = train_mask.sum()
    n_vl = val_mask.sum()

    print(f"Fold {fold_num}: train={n_tr:,}  val={n_vl:,}  RAM={ram_mb():.0f}MB")

    # Val copy (nhỏ ~10%)
    X_vl = X_arr[val_mask].copy()
    y_vl = y_arr[val_mask].copy()
    w_vl = w_arr[val_mask].copy()

    # Dataset — max_bin trong params Dataset, KHÔNG gọi .construct() thủ công
    ds_tr = lgb.Dataset(
        X_arr[train_mask], label=y_arr[train_mask], weight=w_arr[train_mask],
        feature_name=ALL_FEAT, categorical_feature=cat_indices,
        params=DATASET_PARAMS,
        free_raw_data=True,
    )
    ds_vl = lgb.Dataset(
        X_vl, label=y_vl, weight=w_vl,
        feature_name=ALL_FEAT, categorical_feature=cat_indices,
        params=DATASET_PARAMS,
        reference=ds_tr,
        free_raw_data=True,
    )

    callbacks = [
        lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
        lgb.log_evaluation(period=300),
    ]

    try:
        fold_model = lgb.train(
            LGB_PARAMS, ds_tr,
            num_boost_round=NUM_BOOST_ROUND,
            valid_sets=[ds_tr, ds_vl],
            valid_names=['train', 'val'],
            callbacks=callbacks,
        )
    except lgb.basic.LightGBMError as e:
        print(f"  GPU error: {e} → CPU")
        LGB_PARAMS['device_type'] = 'cpu'
        fold_model = lgb.train(
            LGB_PARAMS, ds_tr,
            num_boost_round=NUM_BOOST_ROUND,
            valid_sets=[ds_tr, ds_vl],
            valid_names=['train', 'val'],
            callbacks=callbacks,
        )

    vl_pred    = fold_model.predict(X_vl, num_iteration=fold_model.best_iteration)
    vl_pred_cl = np.clip(vl_pred, CLIP_LOW, CLIP_HIGH)
    fold_score = weighted_rmse_score(y_vl, vl_pred_cl, w_vl)
    baseline   = weighted_rmse_score(y_vl, np.zeros_like(y_vl), w_vl)

    fold_best_iters.append(fold_model.best_iteration)
    fold_val_scores.append(fold_score)

    ok = fold_score > baseline
    print(f"  best_iter={fold_model.best_iteration}  score={fold_score:.6f}  baseline={baseline:.6f}  {'OK' if ok else 'WARN'}")

    del ds_tr, ds_vl, fold_model, X_vl, y_vl, w_vl, vl_pred, vl_pred_cl
    gc.collect()
    print(f"  RAM freed: {ram_mb():.0f} MB\n")

print("=" * 55)
print("WALK-FORWARD RESULTS")
print("=" * 55)
for i, (sc, bi) in enumerate(zip(fold_val_scores, fold_best_iters)):
    label = '← Public LB estimate' if i == len(WF_FOLDS)-1 else ''
    print(f"  Fold {i+1}: score={sc:.6f}  best_iter={bi}  {label}")
print(f"  Mean  : {np.mean(fold_val_scores):.6f}  ← Private LB estimate")
print(f"  Std   : {np.std(fold_val_scores):.6f}  ← nhỏ = ổn định")
FINAL_BEST_ITER = int(np.median(fold_best_iters))
print(f"  Median best_iter: {FINAL_BEST_ITER}")
print("=" * 55)

## Cell 11 — Final Model (train 90%, val 10%)

In [ ]:
print(f"Training FINAL model...  RAM: {ram_mb():.0f} MB")
print(f"rounds = {FINAL_BEST_ITER} (median WF) + early stop safety")

ts_final_split = ts_min + ts_span * WF_FOLDS[-1][0]   # 90%
final_tr_mask  = ts_arr <= ts_final_split
final_vl_mask  = ts_arr > ts_final_split

X_final_val = X_arr[final_vl_mask].copy()
y_final_val = y_arr[final_vl_mask].copy()
w_final_val = w_arr[final_vl_mask].copy()

print(f"final train: {final_tr_mask.sum():,}  val: {final_vl_mask.sum():,}")

ds_final_tr = lgb.Dataset(
    X_arr[final_tr_mask], label=y_arr[final_tr_mask], weight=w_arr[final_tr_mask],
    feature_name=ALL_FEAT, categorical_feature=cat_indices,
    params=DATASET_PARAMS,
    free_raw_data=True,
)
ds_final_vl = lgb.Dataset(
    X_final_val, label=y_final_val, weight=w_final_val,
    feature_name=ALL_FEAT, categorical_feature=cat_indices,
    params=DATASET_PARAMS,
    reference=ds_final_tr,
    free_raw_data=True,
)

try:
    model = lgb.train(
        LGB_PARAMS, ds_final_tr,
        num_boost_round=FINAL_BEST_ITER + 50,
        valid_sets=[ds_final_tr, ds_final_vl],
        valid_names=['train', 'val'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=True),
            lgb.log_evaluation(period=200),
        ],
    )
except lgb.basic.LightGBMError as e:
    print(f"GPU error: {e} → CPU")
    LGB_PARAMS['device_type'] = 'cpu'
    model = lgb.train(
        LGB_PARAMS, ds_final_tr,
        num_boost_round=FINAL_BEST_ITER + 50,
        valid_sets=[ds_final_tr, ds_final_vl],
        valid_names=['train', 'val'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=True),
            lgb.log_evaluation(period=200),
        ],
    )

del ds_final_tr, ds_final_vl
gc.collect()

print(f"Done! best_iter={model.best_iteration}  val_rmse={model.best_score['val']['rmse']:.8f}  RAM:{ram_mb():.0f}MB")

## Cell 12 — Đánh giá Validation

In [ ]:
val_pred    = model.predict(X_final_val, num_iteration=model.best_iteration)
val_pred_cl = np.clip(val_pred, CLIP_LOW, CLIP_HIGH)
baseline_v  = weighted_rmse_score(y_final_val, np.zeros_like(y_final_val), w_final_val)
score_final = weighted_rmse_score(y_final_val, val_pred_cl, w_final_val)

print("=" * 55)
print("FINAL VALIDATION RESULTS")
print("=" * 55)
for i, sc in enumerate(fold_val_scores):
    print(f"  WF fold {i+1}      : {sc:.6f}")
print(f"  WF mean (Private LB est.) : {np.mean(fold_val_scores):.6f}")
print(f"  Final val score           : {score_final:.6f}")
print(f"  Baseline (pred=0)         : {baseline_v:.6f}")
print()
R = 1 - score_final**2
print(f"  Pred range : [{val_pred_cl.min():.4f}, {val_pred_cl.max():.4f}]")
print(f"  % negative : pred={100*(val_pred_cl<0).mean():.1f}%  target={100*(y_final_val<0).mean():.1f}%")
print(f"  Ratio R    : {R:.4f}  (tốt hơn baseline {(1-R)*100:.2f}%)")
print("=" * 55)

del X_arr, y_arr, w_arr, ts_arr
del X_final_val, y_final_val, w_final_val, val_pred, val_pred_cl
gc.collect()
print(f"RAM after freeing all train arrays: {ram_mb():.0f} MB")

## Cell 13 — Feature Importance

In [ ]:
fi = pd.DataFrame({
    'feature'   : model.feature_name(),
    'importance': model.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False).reset_index(drop=True)
print("Top 30 features by GAIN:")
print(fi.head(30).to_string(index=False))

## Cell 14 — Load Test + Predict (Batch)

In [ ]:
print(f"Loading test.parquet...  RAM: {ram_mb():.0f} MB")

test_schema = set(pl.scan_parquet(TEST_PATH).schema.keys())
TEST_NEEDED = [c for c in META_COLS + FEATURE_COLS if c not in [TARGET_COL, WEIGHT_COL]]

test = (
    pl.scan_parquet(TEST_PATH)
    .select([c for c in TEST_NEEDED if c in test_schema])
    .with_columns(
        [pl.col(c).fill_null(impute_vals[c]).fill_nan(impute_vals[c]).cast(pl.Float32)
         for c in FEATURE_COLS] +
        [pl.col('horizon').cast(pl.Int16), pl.col('ts_index').cast(pl.Int32)]
    )
    .collect()
)

for col in CAT_COLS:
    enc = col + '_enc'
    if col in test.columns:
        test = test.with_columns(
            pl.col(col).cast(pl.Utf8).replace(cat_maps[col], default=-1).cast(pl.Int16).alias(enc)
        )

test_ids = test['id'].to_numpy()
X_test   = test.select(ALL_FEAT).to_numpy(allow_copy=True).astype(np.float32)

del test; gc.collect()
print(f"Test: {X_test.shape}  {X_test.nbytes/1e6:.0f}MB  |  RAM: {ram_mb():.0f} MB")
status = 'OK' if len(test_ids) == EXPECTED_TEST_ROWS else 'MISMATCH'
print(f"Rows: {len(test_ids):,} / {EXPECTED_TEST_ROWS:,}  [{status}]")

In [ ]:
print(f"Predicting {len(test_ids):,} rows in batches of {PRED_BATCH_SIZE:,}...")

n_rows    = X_test.shape[0]
preds     = np.empty(n_rows, dtype=np.float32)
n_batches = (n_rows + PRED_BATCH_SIZE - 1) // PRED_BATCH_SIZE

for i in range(n_batches):
    s = i * PRED_BATCH_SIZE
    e = min(s + PRED_BATCH_SIZE, n_rows)
    preds[s:e] = model.predict(X_test[s:e], num_iteration=model.best_iteration).astype(np.float32)
    if (i + 1) % 5 == 0 or i == n_batches - 1:
        print(f"  Batch {i+1}/{n_batches}  RAM: {ram_mb():.0f} MB")

del X_test; gc.collect()

preds_raw     = preds.copy()
preds_clipped = np.clip(preds, CLIP_LOW, CLIP_HIGH)
n_changed     = int((preds_raw != preds_clipped).sum())

print(f"Raw    : min={preds_raw.min():.4f}  max={preds_raw.max():.4f}  std={preds_raw.std():.4f}")
print(f"Clipped: min={preds_clipped.min():.4f}  max={preds_clipped.max():.4f}")
print(f"Changed by clip: {n_changed:,} ({100*n_changed/len(preds):.3f}%)")
print(f"% negative predictions: {100*(preds_clipped<0).mean():.1f}%")

del preds_raw; gc.collect()

## Cell 15 — Tạo Submission

In [ ]:
submission = pd.DataFrame({'id': test_ids, 'prediction': preds_clipped})
submission.to_csv(OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH}")
print(f"Rows : {len(submission):,}")
print(f"NaN  : {submission['prediction'].isna().sum()}")
print(f"Inf  : {np.isinf(submission['prediction'].values).sum()}")
print()
print(submission.head(8).to_string(index=False))

## Cell 16 — Sanity Checks

In [ ]:
print("=" * 65)
print("FINAL SANITY CHECKS")
print("=" * 65)
pv   = submission['prediction'].values
ok1  = len(submission) == EXPECTED_TEST_ROWS
ok2  = not np.isnan(pv).any()
ok3  = not np.isinf(pv).any()

print(f"[{'OK' if ok1 else 'FAIL'}] Rows    : {len(submission):,} / {EXPECTED_TEST_ROWS:,}")
print(f"[{'OK' if ok2 else 'FAIL'}] No NaN")
print(f"[{'OK' if ok3 else 'FAIL'}] No Inf")
print()
print(f"Pred range     : [{pv.min():.4f}, {pv.max():.4f}]")
print(f"Pred mean/std  : {pv.mean():.6f} / {pv.std():.6f}")
print(f"% negative     : {100*(pv<0).mean():.1f}%")
print()
print(f"WF scores      : {[f'{s:.4f}' for s in fold_val_scores]}")
print(f"WF mean        : {np.mean(fold_val_scores):.6f}  <- Private LB estimate")
print(f"WF fold-3      : {fold_val_scores[-1]:.6f}  <- Public LB estimate")
print(f"Final val      : {score_final:.6f}")
print(f"best_iter      : {model.best_iteration}  (WF median: {FINAL_BEST_ITER})")
print("=" * 65)
if all([ok1, ok2, ok3]):
    print("READY TO SUBMIT")
else:
    print("ISSUES FOUND")

## Cell 17 — (Tùy chọn) Retrain 100% data

In [ ]:
RETRAIN_FULL = False   # Đổi True sau khi confirm score OK

if RETRAIN_FULL:
    print(f"Retraining on 100% data for {FINAL_BEST_ITER} rounds...  RAM: {ram_mb():.0f}MB")

    train_f = (
        pl.scan_parquet(TRAIN_PATH)
        .select(NEEDED_COLS)
        .with_columns(
            [pl.col(c).fill_null(impute_vals[c]).fill_nan(impute_vals[c]).cast(pl.Float32) for c in FEATURE_COLS] +
            [pl.col(TARGET_COL).cast(pl.Float32), pl.col(WEIGHT_COL).cast(pl.Float32),
             pl.col('horizon').cast(pl.Int16), pl.col('ts_index').cast(pl.Int32)]
        )
        .collect()
    )
    for col in CAT_COLS:
        train_f = train_f.with_columns(
            pl.col(col).cast(pl.Utf8).replace(cat_maps[col], default=-1).cast(pl.Int16).alias(col + '_enc')
        )
    X_f = train_f.select(ALL_FEAT).to_numpy(allow_copy=True).astype(np.float32)
    y_f = train_f[TARGET_COL].to_numpy().astype(np.float32)
    w_f = train_f[WEIGHT_COL].to_numpy().astype(np.float32)
    del train_f; gc.collect()

    ds_f = lgb.Dataset(X_f, label=y_f, weight=w_f,
                       feature_name=ALL_FEAT, categorical_feature=cat_indices,
                       params=DATASET_PARAMS, free_raw_data=True)
    del X_f, y_f, w_f; gc.collect()

    model_full = lgb.train(LGB_PARAMS, ds_f,
                           num_boost_round=FINAL_BEST_ITER,
                           callbacks=[lgb.log_evaluation(200)])
    del ds_f; gc.collect()

    test_f_schema = set(pl.scan_parquet(TEST_PATH).schema.keys())
    test_f = (
        pl.scan_parquet(TEST_PATH)
        .select([c for c in TEST_NEEDED if c in test_f_schema])
        .with_columns(
            [pl.col(c).fill_null(impute_vals[c]).fill_nan(impute_vals[c]).cast(pl.Float32) for c in FEATURE_COLS] +
            [pl.col('horizon').cast(pl.Int16), pl.col('ts_index').cast(pl.Int32)]
        )
        .collect()
    )
    for col in CAT_COLS:
        if col in test_f.columns:
            test_f = test_f.with_columns(
                pl.col(col).cast(pl.Utf8).replace(cat_maps[col], default=-1).cast(pl.Int16).alias(col + '_enc')
            )
    ids_f  = test_f['id'].to_numpy()
    X_tf   = test_f.select(ALL_FEAT).to_numpy(allow_copy=True).astype(np.float32)
    del test_f; gc.collect()

    preds_f = np.empty(len(ids_f), dtype=np.float32)
    for i in range((len(ids_f) + PRED_BATCH_SIZE - 1) // PRED_BATCH_SIZE):
        s, e = i * PRED_BATCH_SIZE, min((i+1) * PRED_BATCH_SIZE, len(ids_f))
        preds_f[s:e] = model_full.predict(X_tf[s:e]).astype(np.float32)
    del X_tf; gc.collect()

    preds_f = np.clip(preds_f, CLIP_LOW, CLIP_HIGH)
    out_f   = OUTPUT_PATH.replace('.csv', '_100pct.csv')
    pd.DataFrame({'id': ids_f, 'prediction': preds_f}).to_csv(out_f, index=False)
    print(f"Saved: {out_f}  ({len(ids_f):,} rows)")
else:
    print('Skipped (RETRAIN_FULL=False)')